In [15]:
import torch
import numpy as np
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv, LinAlgError
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score
from joblib import Parallel, delayed
import os
import time
TOPICS = {
    "Baseline": None,

    # =========================
    # IMMIGRATION
    # =========================
    "imm_unauth": "unauthorized and undocumented immigrants",
    "birthright": "birthright citizenship",
    "illeg_child": "children of undocumented immigrants",
    "border_wall": "a wall along the U.S.–Mexico border",
    "spend_border": "whether to increase or decrease funding for border security",

    # =========================
    # LABOR / FAMILY POLICY
    # =========================
    "paid_leave": "paid parental or family leave",
    "job_gov_guar": "whether the government should guarantee a job to everyone",
    "min_wage": "whether to change the minimum wage",
    "ft_union": "labor unions",

    # =========================
    # INSTITUTIONS / ACCOUNTABILITY / MEDIA
    # =========================
    "trump_corr": "whether Donald Trump was involved in political corruption",
    "journ_access": "whether journalists should have broad access to government officials and information",
    "checks_power": "to what degree the branches of government should limit each other’s power",
    "rus_interf": "how serious a problem Russian interference in U.S. elections is",

    # =========================
    # ELECTIONS / DEMOCRACY
    # =========================
    "voter_id": "whether voters should be required to show identification to vote",
    "felon_vote": "whether people with felony convictions should have the right to vote",

    # =========================
    # ECONOMY (RETROSPECTIVE)
    # =========================
    "econ_now": "how good or bad the current national economy is",

    # =========================
    # GUNS
    # =========================
    "gun_bkg_chk": "whether background checks should be required for gun purchases",
    "ar_ban": "whether assault weapons should be banned",
    "gun_imp": "to what degree gun regulation is an important political issue",

    # =========================
    # CRIME / POLICING / ORDER
    # =========================
    "death_pen": "whether the death penalty should be used for serious crimes",
    "police_force": "to what degree police should be allowed to use force",
    "urban_unrest": "how serious a problem urban unrest and protests are",
    "ft_police": "the police",

    # =========================
    # ABORTION / COURTS
    # =========================
    "abortion": "whether abortion should be legal",
    "scotus_abort": "the Supreme Court decisions related to abortion",

    # =========================
    # HEALTH
    # =========================
    "govt_health": "whether the government should provide health insurance",
    "obamacare": "whether the Affordable Care Act should be kept, expanded, or repealed",
    "vax_school": "whether children should be required to be vaccinated to attend school",

    # =========================
    # EDUCATION / SPENDING
    # =========================
    "spend_school": "to what degree the federal government should spend money on public schools",

    # =========================
    # REDISTRIBUTION / WELFARE / TAX
    # =========================
    "spend_welfare": "to what degree the federal government should spend money on welfare programs",
    "spend_poor": "to what degree the federal government should spend money to help the poor",
    "svc_spend": "to what degree the government should spend money on public services",
    "millionaire_tax": "whether to tax on millionaires",

    # =========================
    # RACE / DIVERSITY
    # =========================
    "assist_black": "to what degree the government should help Black Americans",
    "black_favor": "whether Black people should get special favor",
    "diversity": "to what degree diversity benefits the country",

    # =========================
    # LGBTQ (SEXUAL ORIENTATION / GENDER IDENTITY)
    # =========================
    "trans_bath": "whether transgender people should be allowed to use bathrooms matching their gender identity",
    "trans_military": "whether transgender people should be allowed to serve in the U.S. military",
    "lg_job": "whether gay and lesbian people should be protected from job discrimination",
    "lg_marry": "whether same-sex couples should be allowed to marry legally",
    "lg_refuse_service": "whether businesses should be allowed to refuse service to same-sex couples",

    # =========================
    # CLIMATE / ENVIRONMENT
    # =========================
    "clim_imp": "how important climate change is as an issue",
    "env_bus": "the tradeoff between environmental protection and business interests",
    "ghg_emiss": "whether to regulate greenhouse gas emissions",

    # =========================
    # DEFENSE / FOREIGN POLICY
    # =========================
    "def_spend": "to what degree the government should spend money on national defense",
    "mil_force": "whether the United States should use military force in foreign  countries",

    # =========================
    # TRADE
    # =========================
    "free_trade": "whether to free trade agreements with other countries",
    "intl_trade_job": "whether international trade helps or hurts jobs in the United States",

    # =========================
    # FEELING THERMOMETERS (IDENTITY / GROUP AFFECT)
    # =========================
    "ft_trans": "transgender people",
    "ft_fem": "feminists",
    "ft_sci": "scientists",
}

SYSTEM_MSG = (
    "You are simulating the public stance of U.S. politicians.\n\n"
)

# ==========================================
# 1. CONFIGURATION
# ==========================================
# A100 40GB can handle large batches for 8B models
BATCH_SIZE = 128
MODEL_PATH = "/project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct"
NOMINATE_CSV = "/project/jevans/maxzhuyt/data/HS116_members_fullname.csv"

# Topics: Core + Bipartisan/Horseshoe Candidates

# ==========================================
# 2. MODEL LOADER & EXTRACTION (GPU)
# ==========================================
def load_model(path):
    print(f"Loading model from: {path}...")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True, local_files_only=True)
    
    # CRITICAL: Left padding allows batching without destroying the last token position
    tokenizer.padding_side = 'left' 
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    model = AutoModelForCausalLM.from_pretrained(
        path, dtype=dtype, device_map="auto", local_files_only=True, attn_implementation="eager"
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

@torch.no_grad()
def extract_heads_batched(model, tokenizer, texts, batch_size=32):
    """
    Optimized extraction for A100.
    """
    model.eval()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads
    D_head = model.config.hidden_size // H
    
    activations = []
    
    # Pre-allocate hook containers
    layer_outputs = [None] * L
    
    def get_hook(layer_idx):
        def hook(module, input, output):
            # Input[0] shape: [Batch, Seq, Hidden]
            # Reshape to [Batch, Seq, Heads, Head_Dim]
            # We immediately move to CPU to free VRAM for the next batch
            reshaped = input[0].detach().view(input[0].shape[0], input[0].shape[1], H, D_head)
            layer_outputs[layer_idx] = reshaped[:, -1, :, :].float().cpu().numpy()
        return hook

    # Register hooks once
    hooks = []
    for li in range(L):
        hooks.append(model.model.layers[li].self_attn.o_proj.register_forward_hook(get_hook(li)))

    print(f"  > Extracting with Batch Size {batch_size}...")
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Fast Tokenization
        formatted_batch = [
            tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": t}], tokenize=False, add_generation_prompt=True)
            for t in batch
        ]
        
        enc = tokenizer(
            formatted_batch, return_tensors="pt", padding=True, truncation=True, max_length=128
        ).to(model.device)
        
        # Forward pass triggers hooks
        model(**enc)
        
        # Stack layers: [Batch, Layers, Heads, Dim]
        batch_acts = np.stack(layer_outputs, axis=1)
        activations.append(batch_acts)

    for h in hooks: h.remove()
    return np.concatenate(activations, axis=0)

# ==========================================
# 3. PARALLEL METRICS ENGINE (CPU)
# ==========================================
def calculate_metrics_for_single_head(head_data, party_labels):
    """
    Calculates all 5 metrics for a SINGLE head.
    This function will be mapped across 1024 heads in parallel.
    """
    # Filter Valid Data
    valid_mask = np.isin(party_labels, [100, 200])
    X = head_data[valid_mask]
    y = party_labels[valid_mask]
    
    # Centering for PCA/Covariance
    X_centered = X - np.mean(X, axis=0)
    
    results = {}
    
    # --- BLOCK A: PCA BASED METRICS (3 in 1) ---
    # We run PCA once to get eigenvalues, used for Dispersion, PC1, and Intrinsic Dim
    try:
        pca = PCA(n_components=10) # We only need top eigenvalues
        pca.fit(X_centered)
        evals = pca.explained_variance_
        
        # 3. Total Dispersion (Sum of variance/eigenvalues)
        results['Total_Dispersion'] = np.sum(evals)
        
        # 4. Explained Variance Ratio of PC1
        results['PC1_Ratio'] = pca.explained_variance_ratio_[0]
        
        # 5. Intrinsic Dimensionality (Participation Ratio)
        sum_evals = np.sum(evals)
        sum_sq_evals = np.sum(evals**2)
        if sum_sq_evals > 0:
            results['Intrinsic_Dim'] = (sum_evals**2) / sum_sq_evals
        else:
            results['Intrinsic_Dim'] = 0.0
            
    except Exception:
        results['Total_Dispersion'] = 0.0
        results['PC1_Ratio'] = 0.0
        results['Intrinsic_Dim'] = 0.0

    # --- BLOCK B: CLUSTER METRICS ---
    
    # 2. Davies-Bouldin Index
    # (Lower is better separation, so higher polarization means LOWER score usually)
    try:
        if len(np.unique(y)) > 1:
            results['Davies_Bouldin'] = davies_bouldin_score(X, y)
        else:
            results['Davies_Bouldin'] = 10.0 # Bad score
    except:
        results['Davies_Bouldin'] = 10.0

    # 1. Mahalanobis Distance
    try:
        dems = X[y == 100]
        reps = X[y == 200]
        
        if len(dems) > 5 and len(reps) > 5:
            # Pooled Covariance with regularization
            cov_pool = (np.cov(dems, rowvar=False) + np.cov(reps, rowvar=False)) / 2
            cov_pool += np.eye(cov_pool.shape[0]) * 1e-6 # Regularize
            
            inv_cov = inv(cov_pool)
            mu_d, mu_r = np.mean(dems, axis=0), np.mean(reps, axis=0)
            results['Mahalanobis'] = mahalanobis(mu_d, mu_r, inv_cov)
        else:
            results['Mahalanobis'] = 0.0
    except (LinAlgError, ValueError):
        results['Mahalanobis'] = 0.0
        
    return results

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================

# Setup
model, tokenizer = load_model(MODEL_PATH)

df_nom = pd.read_csv(NOMINATE_CSV)
df_nom = df_nom[df_nom['party_code'].isin([100, 200])].dropna(subset=['bioname'])
party_labels = df_nom['party_code'].values

full_results = []

print(f"\nStarting Optimized Pipeline on A100 (Batch Size {BATCH_SIZE})")
print(f"Parallel processing enabled for metrics calculation.")

for topic_name, topic_desc in TOPICS.items():
    t0 = time.time()
    print(f"\n--- Topic: {topic_name} ---")
    
    # 1. Generate Prompts
    prompts = []
    for name in df_nom['bioname']:
        if topic_desc:
            user_msg = (
                f"Context: United States, in 2020.\n\n"
                f"Generate a statement by {name} expressing this politician’s position on {topic_desc}."
            )
        else:
            user_msg = (
                f"Context: United States, in 2020.\n\n"
                f"Generate a statement by {name}."
            )
        prompts.append(user_msg)
        
    # 2. Extract Heads (GPU Bound)
    # Shape: [N, 32, 32, 128]
    X_heads = extract_heads_batched(model, tokenizer, prompts, batch_size=BATCH_SIZE)
    
    # 3. Parallel Metrics (CPU Bound)
    # Flatten L and H to iterate easily: List of (1024) arrays of shape (N, 128)
    N, L, H, D = X_heads.shape
    flat_heads = [X_heads[:, l, h, :] for l in range(L) for h in range(H)]
    
    print("  > Computing metrics for 1024 heads (Parallel)...")
    
    # Uses all available CPU cores
    metrics_flat = Parallel(n_jobs=-1)(
        delayed(calculate_metrics_for_single_head)(head_data, party_labels) 
        for head_data in flat_heads
    )
    
    # 4. Aggregation & Storage
    # Reshape back to (32, 32) grids for visualization
    metric_grids = {k: np.zeros((L, H)) for k in metrics_flat[0].keys()}
    
    idx = 0
    for l in range(L):
        for h in range(H):
            m = metrics_flat[idx]
            for key in m:
                metric_grids[key][l, h] = m[key]
            idx += 1
            
    # Calculate Averages/Max for Summary
    summary = {"Topic": topic_name}
    for key, grid in metric_grids.items():
        summary[f"Avg_{key}"] = np.mean(grid)
        summary[f"Max_{key}"] = np.max(grid)
        # Store the full grid for heatmaps later
        summary[f"Grid_{key}"] = grid
        
    full_results.append(summary)
    print(f"  > Done in {time.time() - t0:.1f}s. Avg Mahalanobis: {summary['Avg_Mahalanobis']:.4f}")

# ==========================================
# 5. VISUALIZATION
# ==========================================
df_res = pd.DataFrame(full_results)

Loading model from: /project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct...


Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.65s/it]



Starting Optimized Pipeline on A100 (Batch Size 128)
Parallel processing enabled for metrics calculation.

--- Topic: Baseline ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 13.3s. Avg Mahalanobis: 1.7423

--- Topic: imm_unauth ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 8.6s. Avg Mahalanobis: 1.8204

--- Topic: birthright ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 8.5s. Avg Mahalanobis: 1.7835

--- Topic: illeg_child ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 8.7s. Avg Mahalanobis: 1.8166

--- Topic: border_wall ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 9.1s. Avg Mahalanobis: 1.8052

--- Topic: spend_border ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 head

In [16]:
df_res = pd.DataFrame(full_results)


In [17]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
def adjust_text_positions(x, y, text_list, max_iterations=100, step_size=0.01):
    """
    Iteratively moves text away from:
      1. Its own data point (repulsion)
      2. Other text labels (collision avoidance)
    Returns optimized x_text, y_text arrays.
    """
    n = len(x)
    # Start text positions exactly at data points
    tx = np.array(x, dtype=float)
    ty = np.array(y, dtype=float)
    
    # Normalize coordinates to 0-1 scale for consistent 'force' calculations
    # We will map them back later.
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    # Avoid division by zero
    if x_range == 0: x_range = 1
    if y_range == 0: y_range = 1
    
    tx_norm = (tx - x_min) / x_range
    ty_norm = (ty - y_min) / y_range
    x_norm = (np.array(x) - x_min) / x_range
    y_norm = (np.array(y) - y_min) / y_range

    for _ in range(max_iterations):
        # Calculate forces
        grad_x = np.zeros(n)
        grad_y = np.zeros(n)
        
        for i in range(n):
            # 1. Force pushing text away from its own data point
            # We want it slightly offset, not ON TOP
            dist_self_x = tx_norm[i] - x_norm[i]
            dist_self_y = ty_norm[i] - y_norm[i]
            dist_sq = dist_self_x**2 + dist_self_y**2
            
            # If too close to dot, push away (standard radius)
            target_radius = 0.04 # 4% of plot width
            if dist_sq < target_radius**2:
                 # Push randomly if exactly on top, otherwise radially
                if dist_sq == 0:
                    grad_x[i] += (np.random.random() - 0.5) * 0.1
                    grad_y[i] += (np.random.random() - 0.5) * 0.1
                else:
                    force = (target_radius - np.sqrt(dist_sq)) 
                    grad_x[i] += force * (dist_self_x / np.sqrt(dist_sq))
                    grad_y[i] += force * (dist_self_y / np.sqrt(dist_sq))

            # 2. Force pushing text away from OTHER labels
            for j in range(n):
                if i == j: continue
                
                diff_x = tx_norm[i] - tx_norm[j]
                diff_y = ty_norm[i] - ty_norm[j]
                dist_sq = diff_x**2 + diff_y**2
                
                # Collision radius (approximate text box size)
                min_dist = 0.05 # 5% of plot width
                
                if dist_sq < min_dist**2:
                     if dist_sq == 0:
                        grad_x[i] += (np.random.random() - 0.5) * 0.1
                        grad_y[i] += (np.random.random() - 0.5) * 0.1
                     else:
                        force = (min_dist - np.sqrt(dist_sq)) * 2 # Stronger force for text collision
                        grad_x[i] += force * (diff_x / np.sqrt(dist_sq))
                        grad_y[i] += force * (diff_y / np.sqrt(dist_sq))
                        
        # Apply movements
        tx_norm += grad_x * step_size
        ty_norm += grad_y * step_size
        
        # Clamp to 0-1 to keep inside plot (optional)
        tx_norm = np.clip(tx_norm, 0, 1)
        ty_norm = np.clip(ty_norm, 0, 1)

    # Convert back to original scale
    final_tx = tx_norm * x_range + x_min
    final_ty = ty_norm * y_range + y_min
    
    return final_tx, final_ty

def plot(df_plot):
    # Run the physics engine to get new coordinates for the TEXT ONLY
    # (Assuming adjust_text_positions is defined in your environment)
    new_x, new_y = adjust_text_positions(
        df_plot["Avg_Mahalanobis"].values, 
        df_plot["Avg_Total_Dispersion"].values, 
        df_plot["Topic"].values,
        max_iterations=200, 
        step_size=0.5
    )

    df_plot['text_x'] = new_x
    df_plot['text_y'] = new_y

    # --- 3. PLOTTING (Using Graph Objects for Layering) ---

    fig = go.Figure()

    # Layer 1: The Dots (Markers)
    unique_topics = df_plot['Topic'].unique()
    colors = px.colors.qualitative.Plotly 

    for i, topic in enumerate(unique_topics):
        df_sub = df_plot[df_plot['Topic'] == topic]
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=df_sub['Avg_Mahalanobis'],
            y=df_sub['Avg_Total_Dispersion'],
            mode='markers',
            name=topic,
            # Pass Intrinsic Dim as custom data for the hover
            customdata=df_sub['Avg_Intrinsic_Dim'], 
            marker=dict(
                size=14,  # FIXED SIZE for visualization
                color=color,
                opacity=0.85,
                line=dict(width=1, color='DarkSlateGrey')
            ),
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Polarization: %{x:.4f}<br>" +
                "Intensity: %{y:.2f}<br>" +
                "Intrinsic Dim: %{customdata:.2f}<br>" + # Added here
                "<extra></extra>" 
            ),
            text=df_sub['Topic']
        ))

    # Layer 2: The Lines (Connecting dots to labels)
    for i, row in df_plot.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Avg_Mahalanobis'], row['text_x']],
            y=[row['Avg_Total_Dispersion'], row['text_y']],
            mode='lines',
            line=dict(color='grey', width=0.3),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Layer 3: The Text (at new optimized positions)
    fig.add_trace(go.Scatter(
        x=df_plot['text_x'],
        y=df_plot['text_y'],
        mode='text',
        text=df_plot['Topic'],
        textfont=dict(size=11, color='black'),
        showlegend=False,
        hoverinfo='skip'
    ))

    # --- 4. LAYOUT & ANNOTATIONS ---

    fig.update_layout(
        title="<b>What Politicans Say about Everyday Topics</b><br>X: Polarization | Y: Total Disagreement",
        template="plotly_white",
        height=800,
        width=1000,
        xaxis_title="Polarization (Mahalanobis Distance)",
        yaxis_title="Total Discourse Disagreement (Dispersion)",
        showlegend=True
    )

    # Add Quadrants
    mid_x = df_plot['Avg_Mahalanobis'].median()
    mid_y = df_plot['Avg_Total_Dispersion'].median()

    fig.add_hline(y=mid_y, line_dash="dot", line_color="grey", opacity=0.5)
    fig.add_vline(x=mid_x, line_dash="dot", line_color="grey", opacity=0.5)

    # Add Quadrant Labels (Watermarks)
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>High Polarization<br>High Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>Low Polarization<br>High Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>High Polarization<br>Low Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="bottom")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>Low Polarization<br>Low Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="bottom")

    fig.show()
    fig.write_html("us_cultural_geometry_all.html", include_plotlyjs="cdn")

In [18]:
plot(df_res)

In [19]:
df_res

,Topic,Avg_Total_Dispersion,Max_Total_Dispersion,Grid_Total_Dispersion,Avg_PC1_Ratio,Max_PC1_Ratio,Grid_PC1_Ratio,Avg_Intrinsic_Dim,Max_Intrinsic_Dim,Grid_Intrinsic_Dim,Avg_Davies_Bouldin,Max_Davies_Bouldin,Grid_Davies_Bouldin,Avg_Mahalanobis,Max_Mahalanobis,Grid_Mahalanobis,text_x,text_y
0,Baseline,0.047658,1.387509,"[[0.0005941286217421293, 1.7847796698333696e-0...",0.431750,0.947555,"[[0.1026478260755539, 0.17366264760494232, 0.9...",3.595447,9.322110,"[[8.071420669555664, 5.656852722167969, 1.1524...",9.784196,38.962363,"[[10.713334338130528, 10.213109056063004, 32.9...",1.742303,2.734938,"[[1.297426727558818, 0.6436987625692131, 0.190...",1.733407,0.047658
1,imm_unauth,0.025412,0.815556,"[[1.93697887880262e-05, 2.454397690598853e-05,...",0.379067,0.892721,"[[0.09939506649971008, 0.6425905823707581, 0.8...",3.958459,9.607149,"[[8.002354621887207, 1.5098295211791992, 1.206...",7.887046,30.418443,"[[13.097787468712657, 10.108387830447315, 20.3...",1.820362,2.728656,"[[0.674533106974823, 0.44379014107771114, 0.14...",1.830654,0.024731
2,birthright,0.025352,0.834381,"[[3.131995981675573e-05, 2.3274738850886934e-0...",0.386380,0.901198,"[[0.08284518122673035, 0.6163932681083679, 0.8...",3.914912,9.568889,"[[8.534846305847168, 1.5693951845169067, 1.280...",8.718097,31.322303,"[[12.825210765451795, 9.435209463549949, 13.64...",1.783516,2.777187,"[[0.7733861378325337, 0.47390507778262697, 0.2...",1.793586,0.025433
3,illeg_child,0.029972,0.976445,"[[2.038319689745549e-05, 2.411310742900241e-05...",0.400106,0.917224,"[[0.09910338371992111, 0.6317772269248962, 0.8...",3.809451,9.513084,"[[8.019198417663574, 1.536874771118164, 1.1998...",7.716233,24.755611,"[[13.050365265330207, 9.87556210752656, 19.184...",1.816621,2.754288,"[[0.6861284852529614, 0.4500053155688136, 0.15...",1.825399,0.030153
4,border_wall,0.032572,1.196606,"[[6.063900400476996e-06, 2.596677586552687e-05...",0.398038,0.971956,"[[0.3604753315448761, 0.6265034079551697, 0.97...",3.820195,9.563842,"[[3.212019205093384, 1.557142972946167, 1.0551...",8.359323,23.014879,"[[11.164987589465785, 9.364784536960917, 10.78...",1.805198,2.769580,"[[0.3619386973605039, 0.45704631708487903, 0.1...",1.809131,0.031294
5,spend_border,0.023667,0.851360,"[[5.388631052483106e-06, 2.6470048396731727e-0...",0.383813,0.971407,"[[0.35020822286605835, 0.6348047256469727, 0.9...",3.911231,9.594017,"[[3.3033437728881836, 1.5374172925949097, 1.05...",8.731610,23.342132,"[[10.781268008471029, 9.272498602104898, 10.81...",1.756592,2.726768,"[[0.35949118393181106, 0.4569953483459465, 0.1...",1.759441,0.022424
6,paid_leave,0.019677,0.684278,"[[1.6045252777985297e-05, 2.860807944671251e-0...",0.366038,0.914819,"[[0.10040593892335892, 0.6743094325065613, 0.8...",4.038879,9.541095,"[[7.813446521759033, 1.4319865703582764, 1.223...",7.476192,37.319657,"[[11.321281271213909, 9.490259123622698, 23.46...",1.849040,2.798931,"[[0.6753289483994628, 0.4488995848335882, 0.17...",1.857936,0.019677
7,job_gov_guar,0.025874,0.842723,"[[5.784145741927205e-06, 2.5718809411046095e-0...",0.383822,0.973026,"[[0.34024110436439514, 0.6275244951248169, 0.9...",3.927313,9.600849,"[[3.4832119941711426, 1.5574824810028076, 1.05...",8.243511,44.083817,"[[11.547556269153565, 9.092248584730246, 10.44...",1.775823,2.703181,"[[0.3535567044569992, 0.46263836457499347, 0.1...",1.783484,0.026443
8,min_wage,0.021385,0.800602,"[[1.2617268112080637e-05, 3.3692107535898685e-...",0.372432,0.930449,"[[0.13106130063533783, 0.7141687273979187, 0.9...",3.998114,9.593244,"[[6.596046447753906, 1.3524714708328247, 1.120...",8.138109,33.723942,"[[9.86671041196184, 9.15545967369736, 11.39842...",1.791260,2.720502,"[[0.6353513998087199, 0.45614209926056454, 0.2...",1.798753,0.021988
9,ft_union,0.025730,0.888272,"[[4.1560862882761285e-05, 2.325317109352909e-0...",0.384846,0.935298,"[[0.07728436589241028, 0.5990427732467651, 0.8...",3.921824,9.560122,"[[8.615865707397461, 1.6154125928878784, 1.279...",8.421900,27.618786,"[[12.696556749196374, 9.40855

In [45]:
# Drop topics that are strongly related to the 2020 election cycle. 
DROP_TOPICS = {
    "rus_interf",
    "urban_unrest",
    "econ_now",
    "trump_corr",
    "border_wall"
}

df_res = df_res[~df_res["Topic"].isin(DROP_TOPICS)]
# delete all topics with "ft" in the name



In [46]:
df_res

,Topic,Avg_Total_Dispersion,Max_Total_Dispersion,Grid_Total_Dispersion,Avg_PC1_Ratio,Max_PC1_Ratio,Grid_PC1_Ratio,Avg_Intrinsic_Dim,Max_Intrinsic_Dim,Grid_Intrinsic_Dim,Avg_Davies_Bouldin,Max_Davies_Bouldin,Grid_Davies_Bouldin,Avg_Mahalanobis,Max_Mahalanobis,Grid_Mahalanobis,text_x,text_y
0,Baseline,0.047658,1.387509,"[[0.0005941286217421293, 1.7847796698333696e-0...",0.431750,0.947555,"[[0.1026478260755539, 0.17366264760494232, 0.9...",3.595447,9.322110,"[[8.071420669555664, 5.656852722167969, 1.1524...",9.784196,38.962363,"[[10.713334338130528, 10.213109056063004, 32.9...",1.742303,2.734938,"[[1.297426727558818, 0.6436987625692131, 0.190...",1.733407,0.047658
1,imm_unauth,0.025412,0.815556,"[[1.93697887880262e-05, 2.454397690598853e-05,...",0.379067,0.892721,"[[0.09939506649971008, 0.6425905823707581, 0.8...",3.958459,9.607149,"[[8.002354621887207, 1.5098295211791992, 1.206...",7.887046,30.418443,"[[13.097787468712657, 10.108387830447315, 20.3...",1.820362,2.728656,"[[0.674533106974823, 0.44379014107771114, 0.14...",1.830654,0.024731
2,birthright,0.025352,0.834381,"[[3.131995981675573e-05, 2.3274738850886934e-0...",0.386380,0.901198,"[[0.08284518122673035, 0.6163932681083679, 0.8...",3.914912,9.568889,"[[8.534846305847168, 1.5693951845169067, 1.280...",8.718097,31.322303,"[[12.825210765451795, 9.435209463549949, 13.64...",1.783516,2.777187,"[[0.7733861378325337, 0.47390507778262697, 0.2...",1.793586,0.025433
3,illeg_child,0.029972,0.976445,"[[2.038319689745549e-05, 2.411310742900241e-05...",0.400106,0.917224,"[[0.09910338371992111, 0.6317772269248962, 0.8...",3.809451,9.513084,"[[8.019198417663574, 1.536874771118164, 1.1998...",7.716233,24.755611,"[[13.050365265330207, 9.87556210752656, 19.184...",1.816621,2.754288,"[[0.6861284852529614, 0.4500053155688136, 0.15...",1.825399,0.030153
5,spend_border,0.023667,0.851360,"[[5.388631052483106e-06, 2.6470048396731727e-0...",0.383813,0.971407,"[[0.35020822286605835, 0.6348047256469727, 0.9...",3.911231,9.594017,"[[3.3033437728881836, 1.5374172925949097, 1.05...",8.731610,23.342132,"[[10.781268008471029, 9.272498602104898, 10.81...",1.756592,2.726768,"[[0.35949118393181106, 0.4569953483459465, 0.1...",1.759441,0.022424
6,paid_leave,0.019677,0.684278,"[[1.6045252777985297e-05, 2.860807944671251e-0...",0.366038,0.914819,"[[0.10040593892335892, 0.6743094325065613, 0.8...",4.038879,9.541095,"[[7.813446521759033, 1.4319865703582764, 1.223...",7.476192,37.319657,"[[11.321281271213909, 9.490259123622698, 23.46...",1.849040,2.798931,"[[0.6753289483994628, 0.4488995848335882, 0.17...",1.857936,0.019677
7,job_gov_guar,0.025874,0.842723,"[[5.784145741927205e-06, 2.5718809411046095e-0...",0.383822,0.973026,"[[0.34024110436439514, 0.6275244951248169, 0.9...",3.927313,9.600849,"[[3.4832119941711426, 1.5574824810028076, 1.05...",8.243511,44.083817,"[[11.547556269153565, 9.092248584730246, 10.44...",1.775823,2.703181,"[[0.3535567044569992, 0.46263836457499347, 0.1...",1.783484,0.026443
8,min_wage,0.021385,0.800602,"[[1.2617268112080637e-05, 3.3692107535898685e-...",0.372432,0.930449,"[[0.13106130063533783, 0.7141687273979187, 0.9...",3.998114,9.593244,"[[6.596046447753906, 1.3524714708328247, 1.120...",8.138109,33.723942,"[[9.86671041196184, 9.15545967369736, 11.39842...",1.791260,2.720502,"[[0.6353513998087199, 0.45614209926056454, 0.2...",1.798753,0.021988
9,ft_union,0.025730,0.888272,"[[4.1560862882761285e-05, 2.325317109352909e-0...",0.384846,0.935298,"[[0.07728436589241028, 0.5990427732467651, 0.8...",3.921824,9.560122,"[[8.615865707397461, 1.6154125928878784, 1.279...",8.421900,27.618786,"[[12.696556749196374, 9.4085570748373, 18.8068...",1.792478,2.725250,"[[0.8211638510522383, 0.49715459536521917, 0.1...",1.799002,0.026697
11,journ_access,0.024609,0.894632,"[[3.118649601674406e-06, 1.6123845853144303e-0...",0.380107,0.968254,"[[0.18235979974269867, 0.5166677832603455, 0.9...",3.947253,9.596963,"[[5.625801086425781, 1.927033543586731, 1.0609...",8.615296,29.417355,"[[11.88519153513312, 11.149

In [47]:
anes_path = "policy_polarization.csv"

df_anes = pd.read_csv(anes_path)
df_anes = df_anes.rename(columns={"issue": "Topic", "mahalanobis_distance": "ANES_Mahalanobis", "variance": "ANES_Variance"})
df_anes

,Topic,ANES_Mahalanobis,ANES_Variance,area,explanation
0,imm_unauth,1.007885,0.087450,Immigration,Policy preferences regarding unauthorized/undo...
1,birthright,1.036311,0.129074,Immigration,Support for birthright citizenship for childre...
2,paid_leave,0.713889,0.072350,Labor,Support for paid parental/family leave policie...
3,trump_corr,1.637288,0.093275,Institutions,Perceptions of corruption involving Donald Tru...
4,journ_access,0.834154,0.104073,Institutions,Support for press/journalist access to governm...
5,gun_bkg_chk,0.497340,0.050344,Guns,Support for background checks on gun purchases...
6,illeg_child,0.948134,0.075581,Immigration,Views on how the U.S. should handle children o...
7,death_pen,0.999484,0.147272,Crime,Support for the death penalty/capital punishme...
8,abortion,1.232532,0.132176,Abortion,Attitudes toward abortion legality and access.
9,scotus_abort,1.413303,0.120520,Abortion,Views on Supreme Court decisions and judicial ...


## Comparing Party Polarization between LLM and ANES

In [34]:

# Keep only Topic + Avg_Mahalanobis from LLM results
# rename columns

df_llm = df_res[["Topic", "Avg_Mahalanobis", "Avg_Total_Dispersion"]].copy()

# Inner join ensures exact key matching
df_merged = df_llm.merge(
    df_anes,
    left_on="Topic",
    right_on="Topic",
    how="inner"
)

pearson_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="pearson"
)

print(f"Pearson correlation: {pearson_corr:.3f}")
spearman_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="spearman"
)

print(f"Spearman correlation: {spearman_corr:.3f}")
df_merged["LLM_rank"] = df_merged["Avg_Mahalanobis"].rank(ascending=False)
df_merged["ANES_rank"] = df_merged["ANES_Mahalanobis"].rank(ascending=False)

print(df_merged.sort_values("Avg_Mahalanobis", ascending=False).to_markdown())


Pearson correlation: 0.254
Spearman correlation: 0.178
|    | Topic             |   Avg_Mahalanobis |   Avg_Total_Dispersion |   ANES_Mahalanobis |   ANES_Variance | area           | explanation                                                                                       |   LLM_rank |   ANES_rank |
|---:|:------------------|------------------:|-----------------------:|-------------------:|----------------:|:---------------|:--------------------------------------------------------------------------------------------------|-----------:|------------:|
| 33 | lg_job            |           1.88384 |              0.0384717 |           0.602085 |       0.0886719 | LGBTQ          | Views on legal protections against job discrimination for gay/lesbian people.                     |          1 |        38.5 |
| 43 | ft_trans          |           1.87763 |              0.0408137 |           1.06988  |       0.0763283 | LGBTQ          | Feeling thermometer rating toward transgender people

### Can we look at what subset correlate well and try to reverse engineer what might be the issue ...?

In [38]:
# Sample subsets and find best correlations
import random
random.seed(42)
np.random.seed(42)

n_iterations = 10000
sample_size = 25
results = []

for i in range(n_iterations):
    # Sample 20 random issues
    sampled_df = df_merged.sample(n=sample_size, random_state=i)
    
    # Calculate correlation
    pearson = sampled_df["Avg_Mahalanobis"].corr(sampled_df["ANES_Mahalanobis"], method="pearson")
    spearman = sampled_df["Avg_Mahalanobis"].corr(sampled_df["ANES_Mahalanobis"], method="spearman")
    
    results.append({
        'iteration': i,
        'pearson': pearson,
        'spearman': spearman,
        'topics': sampled_df['Topic'].tolist()
    })

# Convert to DataFrame and sort by Pearson correlation
df_results = pd.DataFrame(results)
df_results_sorted = df_results.sort_values('pearson', ascending=False)

# Get top 5
top_10 = df_results_sorted.head(10)

print("Top 10 iterations by Pearson correlation:")
for idx, row in top_10.iterrows():
    print(f"\nIteration {row['iteration']}: Pearson={row['pearson']:.3f}, Spearman={row['spearman']:.3f}")
    print(f"Topics: {', '.join(row['topics'])}")

Top 10 iterations by Pearson correlation:

Iteration 3741: Pearson=0.702, Spearman=0.675
Topics: intl_trade_job, felon_vote, vax_school, black_favor, def_spend, paid_leave, voter_id, trans_military, checks_power, spend_school, trans_bath, assist_black, ft_union, ft_fem, ft_police, diversity, free_trade, ft_sci, mil_force, abortion, lg_refuse_service, clim_imp, obamacare, imm_unauth, journ_access

Iteration 7023: Pearson=0.701, Spearman=0.724
Topics: checks_power, imm_unauth, spend_school, ft_fem, ft_police, clim_imp, def_spend, scotus_abort, min_wage, vax_school, ft_trans, trans_bath, ft_sci, mil_force, ft_union, illeg_child, lg_refuse_service, millionaire_tax, spend_poor, assist_black, trans_military, journ_access, spend_welfare, obamacare, free_trade

Iteration 1701: Pearson=0.696, Spearman=0.576
Topics: checks_power, ft_union, mil_force, abortion, voter_id, spend_school, assist_black, lg_refuse_service, ft_fem, journ_access, ft_sci, ar_ban, ghg_emiss, free_trade, felon_vote, job_gov

In [35]:
def compare_standardized_columns(df, col1, col2, group_by='area'):
    """
    Demean and standardize two columns separately, calculate their difference,
    and group by specified column.
    
    Parameters:
    - df: DataFrame containing the data
    - col1: First column name to standardize
    - col2: Second column name to standardize
    - group_by: Column name to group results by (default 'area')
    
    Returns:
    - Modified DataFrame with normalized columns and difference
    """
    # Create a copy to avoid modifying original
    df_work = df.copy()
    
    # Demean and standardize each column separately
    mean1 = df_work[col1].mean()
    std1 = df_work[col1].std()
    mean2 = df_work[col2].mean()
    std2 = df_work[col2].std()
    
    col1_norm = col1 + '_norm'
    col2_norm = col2 + '_norm'
    
    df_work[col1_norm] = (df_work[col1] - mean1) / std1
    df_work[col2_norm] = (df_work[col2] - mean2) / std2
    
    # Calculate difference between standardized columns
    df_work['diff'] = df_work[col1_norm] - df_work[col2_norm]
    
    # Calculate mean differences by group
    group_diff = df_work.groupby(group_by).agg({
        'diff': ['mean', 'count']
    }).round(4)
    
    group_diff.columns = ['mean_diff', 'n_topics']
    group_diff = group_diff.sort_values('mean_diff', ascending=False)
    
    print(f"Standardized Differences by {group_by}:")
    print(group_diff)
    
    print("\nDetailed breakdown:")
    print(df_work[['Topic', group_by, col1_norm, col2_norm, 'diff']].sort_values('diff', ascending=False).to_markdown())
    
    return df_work

# Compare Avg_Total_Dispersion (LLM) with ANES_Variance
df_merged = compare_standardized_columns(df_merged, 'Avg_Mahalanobis', 'ANES_Mahalanobis', group_by='area')

Standardized Differences by area:
                mean_diff  n_topics
area                               
Guns               2.2399         3
LGBTQ              1.3205         6
Trade              0.4599         2
ForeignPolicy      0.4402         1
Elections          0.2223         2
Labor              0.1659         4
Race              -0.1165         3
Immigration       -0.2141         4
Abortion          -0.3796         2
Education         -0.4103         1
Gender            -0.6406         1
Institutions      -0.6758         3
Redistribution    -0.7144         4
Policing          -0.7302         2
Climate           -0.7316         3
Crime             -0.7631         1
Health            -1.0616         3
Defense           -1.6063         1

Detailed breakdown:
|    | Topic             | area           |   Avg_Mahalanobis_norm |   ANES_Mahalanobis_norm |        diff |
|---:|:------------------|:---------------|-----------------------:|------------------------:|------------:|
| 33 | 

In [54]:
# restrict area to some areas
df_subset = df_merged[df_merged['area'].isin(['Institutions', 'Policing', 'Labor', 'Race', "Immigration", "Abortion", "Education"])]
pearson_corr_subset = df_subset["Avg_Mahalanobis"].corr(
    df_subset["ANES_Mahalanobis"],
    method="pearson"
)
print(f"\nSubset Pearson correlation: {pearson_corr_subset:.3f}")
spearman_corr_subset = df_subset["Avg_Mahalanobis"].corr(
    df_subset["ANES_Mahalanobis"],
    method="spearman"
)
print(f"Subset Spearman correlation: {spearman_corr_subset:.3f}")



Subset Pearson correlation: 0.518
Subset Spearman correlation: 0.305


## Comparing Total Variance between ANES and LLM

In [40]:

pearson_corr = df_merged["Avg_Total_Dispersion"].corr(
    df_merged["ANES_Variance"],
    method="pearson"
)

print(f"Pearson correlation: {pearson_corr:.3f}")
spearman_corr = df_merged["Avg_Total_Dispersion"].corr(
    df_merged["ANES_Variance"],
    method="spearman"
)

print(f"Spearman correlation: {spearman_corr:.3f}")
df_merged["LLM_rank"] = df_merged["Avg_Total_Dispersion"].rank(ascending=False)
df_merged["ANES_rank"] = df_merged["ANES_Variance"].rank(ascending=False)

df_merged.sort_values("Avg_Total_Dispersion", ascending=False)

Pearson correlation: 0.023
Spearman correlation: 0.126


,Topic,Avg_Mahalanobis,Avg_Total_Dispersion,ANES_Mahalanobis,ANES_Variance,area,explanation,LLM_rank,ANES_rank,Avg_Mahalanobis_norm,ANES_Mahalanobis_norm,diff,Avg_Total_Dispersion_norm,ANES_Variance_norm
44,ft_fem,1.797567,0.041223,1.355457,0.071815,Gender,Feeling thermometer rating toward feminists.,1.0,35.0,0.080517,0.721157,3.348773,2.612767,-0.736006
43,ft_trans,1.877632,0.040814,1.069882,0.076328,LGBTQ,Feeling thermometer rating toward transgender ...,2.0,31.0,1.771682,0.076265,3.137853,2.537358,-0.600496
33,lg_job,1.883839,0.038472,0.602085,0.088672,LGBTQ,Views on legal protections against job discrim...,3.0,24.5,1.902781,-0.980125,2.335274,2.105366,-0.229908
32,trans_military,1.841814,0.037917,1.205945,0.104121,LGBTQ,Views on transgender people serving in the U.S...,4.0,19.0,1.015122,0.383524,1.769062,2.002983,0.233921
42,intl_trade_job,1.731186,0.034337,0.153209,0.065719,Trade,Beliefs about international trade’s impact on ...,5.0,38.0,-1.321603,-1.993787,2.261803,1.342776,-0.919027
18,abortion,1.804190,0.032865,1.232532,0.132176,Abortion,Attitudes toward abortion legality and access.,6.0,8.0,0.220404,0.443563,-0.004925,1.071264,1.076189
34,lg_marry,1.843074,0.032455,0.602085,0.088672,LGBTQ,Support for same-sex marriage / marriage right...,7.0,24.5,1.041728,-0.980125,1.225455,0.995547,-0.229908
28,assist_black,1.853854,0.032139,1.623140,0.112476,Race,Support for government assistance to Black Ame...,8.0,14.0,1.269436,1.325645,0.452574,0.937311,0.484737
35,lg_refuse_service,1.828013,0.032093,1.243084,0.165661,LGBTQ,Views on refusing service to same-sex couples.,9.0,1.0,0.723607,0.467393,-1.152795,0.928706,2.081502
36,clim_imp,1.835516,0.032024,1.565396,0.114112,Climate,Importance assigned to climate change as an is...,10.0,12.0,0.882089,1.195245,0.382277,0.916135,0.533858


In [37]:
df_merged = compare_standardized_columns(df_merged, 'Avg_Total_Dispersion', 'ANES_Variance')

Standardized Differences by area:
                mean_diff  n_topics
area                               
Gender             3.3488         1
ForeignPolicy      1.9005         1
Trade              1.4179         2
LGBTQ              0.9298         6
Institutions       0.7470         3
Policing           0.7432         2
Race              -0.0694         3
Labor             -0.1280         4
Abortion          -0.1893         2
Climate           -0.1952         3
Immigration       -0.3251         4
Defense           -0.4532         1
Education         -0.4559         1
Guns              -0.5860         3
Redistribution    -0.8919         4
Elections         -0.9339         2
Health            -1.4083         3
Crime             -2.0789         1

Detailed breakdown:
|    | Topic             | area           |   Avg_Total_Dispersion_norm |   ANES_Variance_norm |        diff |
|---:|:------------------|:---------------|----------------------------:|---------------------:|------------:|
| 4


Subset Pearson correlation: 0.641
Subset Spearman correlation: 0.434
